# MLPC 2026 — Challenge: Sound Event Detection
**Team Temporal Frameworks** — Sebastian Pichler, Laurin Siebert

Extending the segment classifiers from Task 4 to full sound event detection (SED) for recordings of arbitrary length, predicting which of the 15 classes are active and when (onset/offset).

Pipeline mirrors the provided challenge baseline, scoring uses the official segment-based macro F1 from `evaluate.py`.

Task 1 (baseline reproduction) is handled by the provided `challenge_baseline/` notebook, this notebook starts at Task 2.

### What this notebook covers

1. Setup
2. Feature & label loading
3. Data splits
4. SED inference & evaluation helpers
5. Training data
6. Task 2 — simple classifier (our XGBoost from Task 4)
7. Hidden-test submission


## 1. Setup

In [14]:
import os
import sys
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from typing import List, Dict, Tuple, Callable

# evaluate.py is located in the 'provided_baseline' subdirectory
sys.path.insert(0, os.path.join(os.getcwd(), "provided_baseline"))
from evaluate import (
    aggregate_ground_truth_annotations,
    build_segment_frame_from_intervals,
    calculate_f1_score,
)

# copied provided functions from challenge_baseline.ipynb into sed_utils.py for better organization
from sed_utils import (
    build_feature_matrix,
    get_segment_labels,
    load_all_segments,
    run_sed_inference,
    predictions_to_intervals,
    generate_predictions,
    evaluate_split,
)

rng = np.random.default_rng(seed=42)

In [15]:
PATH_TO_DATASET = "../../data/MLPC2026_challenge"

PATH_TRAIN = os.path.join(PATH_TO_DATASET, "train")
PATH_VAL   = os.path.join(PATH_TO_DATASET, "validation")
PATH_TEST  = os.path.join(PATH_TO_DATASET, "test")

# Sanity check — will raise an AssertionError if a path does not exist.
for path in [PATH_TRAIN, PATH_VAL, PATH_TEST]:
    assert os.path.isdir(path), f"Directory not found: {path}"
for path in [
    os.path.join(PATH_TRAIN, "annotations.csv"),
    os.path.join(PATH_TRAIN, "audio_features"),
    os.path.join(PATH_VAL,   "annotations.csv"),
    os.path.join(PATH_VAL,   "audio_features"),
    os.path.join(PATH_TEST,  "audio_features"),
]:
    assert os.path.exists(path), f"Path not found: {path}"

print("Dataset paths OK.")

SEGMENT_LENGTH = 1.0  # each feature vector covers a 1-second window
HOP_SIZE       = 0.5  # segments are extracted with 50% overlap

FEATURE_NAMES = [
    "zcr_mean",        "zcr_std",        "zcr_min",        "zcr_max",
    "melspect_mean",   "melspect_std",   "melspect_min",   "melspect_max",
    "mfcc_mean",       "mfcc_std",       "mfcc_min",       "mfcc_max",
    "mfcc_d_mean",     "mfcc_d_std",     "mfcc_d_min",     "mfcc_d_max",
    "mfcc_d2_mean",    "mfcc_d2_std",    "mfcc_d2_min",    "mfcc_d2_max",
    "flux_mean",       "flux_std",       "flux_min",       "flux_max",
    "flatness_mean",   "flatness_std",   "flatness_min",   "flatness_max",
    "centroid_mean",   "centroid_std",   "centroid_min",   "centroid_max",
    "bandwidth_mean",  "bandwidth_std",  "bandwidth_min",  "bandwidth_max",
    "contrast_mean",   "contrast_std",   "contrast_min",   "contrast_max",
    "rolloff_low_mean",  "rolloff_low_std",  "rolloff_low_min",  "rolloff_low_max",
    "rolloff_high_mean", "rolloff_high_std", "rolloff_high_min", "rolloff_high_max",
    "energy_mean",     "energy_std",     "energy_min",     "energy_max",
    "power_mean",      "power_std",      "power_min",      "power_max",
]

# subset settled on in Task 4 (dropped mel-spectrogram, keep zcr / mfcc(+d, +d2) / spectral contrast / power)
# 420 dims for XGBoost
FEATURE_SELECT = [
    "zcr_mean", "zcr_std", "zcr_min", "zcr_max",
    "mfcc_mean", "mfcc_std", "mfcc_min", "mfcc_max",
    "mfcc_d_mean", "mfcc_d_std", "mfcc_d_min", "mfcc_d_max",
    "mfcc_d2_mean", "mfcc_d2_std", "mfcc_d2_min", "mfcc_d2_max",
    "contrast_mean", "contrast_std", "contrast_min", "contrast_max",
    "power_mean", "power_std", "power_min", "power_max",
]

# The 15 target sound event classes — sorted alphabetically to match the .npz annotation order.
CLASS_NAMES = [
    "bell_ringing",
    "coffee_machine",
    "cutlery_dishes",
    "door_open_close",
    "footsteps",
    "keyboard_typing",
    "keychain",
    "light_switch",
    "microwave",
    "phone_ringing",
    "running_water",
    "toilet_flushing",
    "vacuum_cleaner",
    "wardrobe_drawer_open_close",
    "window_open_close",
]

Dataset paths OK.


## 2. Feature & label loading

Each `.npz` holds per-frame features and an `annotations` array of shape `(T, C, A)` — overlap fraction per frame, class, annotator. We binarise a class as active if any annotator overlapped the frame (`> 0`) and then majority-vote across annotators (same rule as the baseline).

In [16]:
# precompute column indices of the FEATURE_SELECT subset
_sample = dict(np.load(sorted(glob.glob(os.path.join(PATH_TRAIN, "audio_features", "*.npz")))[0],allow_pickle=True))
FEATURE_WIDTH = {n: (_sample[n].shape[1] if _sample[n].ndim > 1 else 1) for n in FEATURE_NAMES}

def feature_columns(feature_names) -> np.ndarray:
    span, c = {}, 0
    for n in FEATURE_NAMES:
        span[n] = (c, c + FEATURE_WIDTH[n]); c += FEATURE_WIDTH[n]
    idx = []
    for n in feature_names:
        idx += list(range(*span[n]))
    return np.array(idx)

SELECT_COLS = feature_columns(FEATURE_SELECT)   # column indices of the 420-dim subset

## 3. Data splits

`train` / `validation` / `test` are predefined. The hidden `test` split has no labels, so — as in the baseline — we split the 999 validation recordings 50/50 into a local validation set (modelselection / tuning) and a non-hidden test set (one-shot final check, not used for tuning).

In [17]:
train_files = sorted(glob.glob(os.path.join(PATH_TRAIN, "audio_features", "*.npz")))
val_files   = sorted(glob.glob(os.path.join(PATH_VAL,   "audio_features", "*.npz")))
test_files  = sorted(glob.glob(os.path.join(PATH_TEST,  "audio_features", "*.npz")))

val_shuffled = rng.permutation(val_files).tolist()
half = len(val_shuffled) // 2
our_val_files  = val_shuffled[:half]   # local validation set
our_test_files = val_shuffled[half:]   # non-hidden local test set

print(f"train:           {len(train_files)}")
print(f"local val:       {len(our_val_files)}")
print(f"non-hidden test: {len(our_test_files)}")
print(f"hidden test:     {len(test_files)}")

# validation-split ground truth, shared by both local evaluations
ann_df = pd.read_csv(os.path.join(PATH_VAL, "annotations.csv"))

train:           3704
local val:       499
non-hidden test: 500
hidden test:     1007
